# 01 — Exploratory Data Analysis

Covers:
1. Database overview — table sizes, stock universe
2. 5-year price history — AAPL, JPM, SPY, sector ETFs
3. Return distributions — daily returns; skew / kurtosis / fat tails
4. Rolling volatility — realized vol over time
5. Cross-asset correlation — return correlation matrix
6. Technical features — current snapshot for available stocks
7. Fundamental features — key ratio comparison AAPL vs JPM
8. Sentiment features — current signals
9. Missingness analysis — per-feature null rates
10. Data coverage status — per-stock table coverage

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

ROOT    = Path("..")
DB_PATH = ROOT / "database" / "stock_dashboard.db"

def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def q(sql, params=()):
    """Run a query and return a DataFrame."""
    with get_conn() as conn:
        return pd.read_sql_query(sql, conn, params=list(params))

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

print(f"DB path: {DB_PATH.resolve()}")
print(f"DB exists: {DB_PATH.exists()}")

## 1. Database Overview

In [ ]:
with get_conn() as conn:
    tables = [r[0] for r in conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()]

counts = [{"table": t, "rows": int(q(f"SELECT COUNT(*) as n FROM {t}")["n"].iloc[0])} for t in tables]
print(pd.DataFrame(counts).to_string(index=False))

In [ ]:
df_stocks = q("""
    SELECT ticker, company_name, sector, sector_etf, is_active,
           (SELECT COUNT(*) FROM daily_prices WHERE stock_id=s.id) as price_rows,
           (SELECT MAX(date)  FROM daily_prices WHERE stock_id=s.id) as latest_price
    FROM stocks s
    ORDER BY is_active DESC, sector, ticker
""")
active_cnt = int((df_stocks["is_active"] == 1).sum())
etf_cnt    = int((df_stocks["is_active"] == 0).sum())
print(f"Total stocks: {len(df_stocks)}  ({active_cnt} active, {etf_cnt} ETF references)")
print()
print(df_stocks.to_string(index=False))

In [ ]:
has_prices = df_stocks[df_stocks["price_rows"] > 0][["ticker","sector","price_rows","latest_price"]]
no_prices  = df_stocks[df_stocks["price_rows"] == 0][["ticker","sector"]]
print(f"With price history ({len(has_prices)}):")
print(has_prices.to_string(index=False))
print()
print(f"WITHOUT price history ({len(no_prices)}) — need daily_collect:")
print(no_prices.to_string(index=False))

## 2. 5-Year Price History

In [ ]:
df_prices = q("""
    SELECT s.ticker, p.date, p.adj_close, p.volume
    FROM daily_prices p
    JOIN stocks s ON s.id = p.stock_id
    WHERE p.adj_close IS NOT NULL
    ORDER BY s.ticker, p.date
""")
df_prices["date"] = pd.to_datetime(df_prices["date"])
print(f"Price rows: {len(df_prices):,}")
print(f"Tickers: {sorted(df_prices['ticker'].unique())}")
print(f"Date range: {df_prices['date'].min().date()} — {df_prices['date'].max().date()}")

In [ ]:
prices_wide = df_prices.pivot(index="date", columns="ticker", values="adj_close")
prices_norm = prices_wide.div(prices_wide.iloc[0]) * 100

highlight = [t for t in ["AAPL", "JPM", "SPY"] if t in prices_norm.columns]
etfs      = [t for t in ["XLK","XLF","XLV","XLE","XLI","XLP","XLY"] if t in prices_norm.columns]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ax = axes[0]
for t in highlight:
    ax.plot(prices_norm.index, prices_norm[t], label=t, linewidth=1.5)
ax.set_title("Indexed Price (100 = start)  |  AAPL, JPM, SPY")
ax.set_ylabel("Index")
ax.legend()
ax.xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%Y"))

ax = axes[1]
for t in etfs:
    ax.plot(prices_norm.index, prices_norm[t], label=t, linewidth=1.2, alpha=0.8)
if "SPY" in prices_norm.columns:
    ax.plot(prices_norm.index, prices_norm["SPY"], label="SPY", color="black", linewidth=2, linestyle="--")
ax.set_title("Sector ETFs vs SPY (Indexed)")
ax.legend(ncol=2, fontsize=8)
ax.xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

total_returns = ((prices_wide.iloc[-1] / prices_wide.iloc[0]) - 1) * 100
print("Total return over full period:")
print(total_returns.sort_values(ascending=False).apply(lambda x: f"{x:+.1f}%").to_string())

## 3. Return Distributions

**Why this matters for ML:** Daily returns have fat tails (leptokurtosis) — extreme moves (-10% days) occur far more often than a Gaussian predicts. The `FeaturePreprocessor` clips outliers to [1%, 99%] before z-scoring to prevent these from distorting normalization for all other samples.

In [ ]:
returns = np.log(prices_wide / prices_wide.shift(1)).dropna()
focus   = [t for t in ["AAPL","JPM","SPY"] if t in returns.columns]

fig, axes = plt.subplots(1, len(focus), figsize=(5*len(focus), 4))
if len(focus) == 1: axes = [axes]

for ax, ticker in zip(axes, focus):
    r = returns[ticker].dropna()
    ax.hist(r, bins=100, density=True, alpha=0.6, color="steelblue", label=ticker)
    x = np.linspace(r.min(), r.max(), 300)
    ax.plot(x, stats.norm.pdf(x, r.mean(), r.std()), "r--", linewidth=1.5, label="Normal")
    ax.set_title(f"{ticker}\nkurt={r.kurtosis():.2f}, skew={r.skew():.2f}")
    ax.set_xlabel("Daily log return")
    ax.legend(fontsize=8)

plt.suptitle("Return Distributions vs Normal  —  fat tails visible", y=1.02)
plt.tight_layout()
plt.show()

stats_df = returns[focus].describe().T
stats_df["ann_vol"]  = returns[focus].std() * np.sqrt(252)
stats_df["skewness"] = returns[focus].skew()
stats_df["kurtosis"] = returns[focus].kurtosis()
print(stats_df[["mean","std","ann_vol","skewness","kurtosis","min","max"]].round(4))

In [ ]:
# QQ Plot — fat tail diagnosis
fig, axes = plt.subplots(1, len(focus), figsize=(5*len(focus), 4))
if len(focus) == 1: axes = [axes]

for ax, ticker in zip(axes, focus):
    r = returns[ticker].dropna()
    (osm, osr), (slope, intercept, _) = stats.probplot(r, dist="norm")
    ax.scatter(osm, osr, s=1, alpha=0.3, color="steelblue")
    ax.plot(osm, slope * np.array(osm) + intercept, "r--", linewidth=1.5)
    ax.set_title(f"{ticker} QQ Plot")
    ax.set_xlabel("Theoretical quantiles (normal)")
    ax.set_ylabel("Sample quantiles")

plt.suptitle("QQ Plots  —  deviation at extremes = fat tails", y=1.02)
plt.tight_layout()
plt.show()

print("Points curving away from the red line at extremes = heavier tails than normal.")
print("Winsorization at 1%/99% addresses this before z-score normalization.")

## 4. Rolling Volatility

Realized volatility clusters — high-vol periods follow high-vol periods (GARCH effect). `realized_vol_20d` and `realized_vol_60d` in `technical_features` capture this regime. `vol_ratio` (20d/60d) signals vol regime transitions: ratio > 1 means current vol is elevated.

In [ ]:
rolling_vol  = returns.rolling(30).std() * np.sqrt(252)
plot_tickers = [t for t in ["AAPL","JPM","SPY"] if t in rolling_vol.columns]

fig, ax = plt.subplots(figsize=(14, 5))
for t in plot_tickers:
    ax.plot(rolling_vol.index, rolling_vol[t], label=t, linewidth=1.3)

ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_title("30-Day Rolling Realized Volatility (Annualized)")
ax.set_ylabel("Annualized Vol")
ax.legend()
ax.xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

print("Current vol vs trailing 2-year percentile:")
for t in plot_tickers:
    v        = rolling_vol[t].dropna()
    current  = v.iloc[-1]
    hist_2yr = v.iloc[-504:].dropna()
    pct      = stats.percentileofscore(hist_2yr, current)
    print(f"  {t}: {current:.1%}  —  {pct:.0f}th percentile of 2yr history")

## 5. Cross-Asset Correlation Matrix

Pearson correlation of daily returns. High within-sector correlation is expected. The ML model needs cross-sector features (macro, sentiment) to differentiate stocks that move together in the short run.

In [ ]:
corr = returns.corr()

sector_order    = ["SPY","XLK","AAPL","XLF","JPM","XLV","XLE","XLI","XLY","XLP"]
present_ordered = [c for c in sector_order if c in corr.columns]
remaining       = [c for c in corr.columns if c not in sector_order]
ordered         = present_ordered + remaining
corr_ordered    = corr.loc[ordered, ordered]

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_ordered, dtype=bool), k=1)
sns.heatmap(
    corr_ordered, mask=mask,
    annot=True, fmt=".2f",
    cmap="RdYlGn", center=0, vmin=-0.2, vmax=1.0,
    square=True, linewidths=0.5,
    annot_kws={"size": 8}, ax=ax
)
ax.set_title("Daily Return Correlation (lower triangle)")
plt.tight_layout()
plt.show()

for a, b in [("AAPL","SPY"),("JPM","SPY"),("AAPL","JPM"),("AAPL","XLK"),("JPM","XLF")]:
    if a in corr.index and b in corr.columns:
        print(f"  {a}–{b}: {corr.loc[a,b]:.3f}")

In [ ]:
# Rolling 90-day correlation vs SPY
if "SPY" in returns.columns:
    fig, ax = plt.subplots(figsize=(14, 4))
    for t in ["AAPL","JPM"]:
        if t in returns.columns:
            roll_corr = returns[t].rolling(90).corr(returns["SPY"])
            ax.plot(roll_corr.index, roll_corr, label=f"{t}–SPY", linewidth=1.3)

    ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
    ax.set_ylim(-0.5, 1.05)
    ax.set_title("90-Day Rolling Correlation vs SPY")
    ax.set_ylabel("Pearson r")
    ax.legend()
    ax.xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%Y"))
    plt.tight_layout()
    plt.show()

## 6. Technical Features — Current Snapshot

In [ ]:
df_tech = q("""
    SELECT s.ticker, t.*
    FROM technical_features t
    JOIN stocks s ON s.id = t.stock_id
    WHERE t.date = (SELECT MAX(date) FROM technical_features)
    ORDER BY s.ticker
""")

skip  = {"id","stock_id","date","created_at"}
fcols = [c for c in df_tech.columns if c not in skip]
print(f"Stocks: {df_tech['ticker'].tolist()}")
print(f"Feature count: {len(fcols)-1}  (exc. ticker)")
print(f"As-of: {df_tech['date'].iloc[0] if len(df_tech) else 'N/A'}")

In [ ]:
if len(df_tech) >= 2:
    show = [f for f in [
        "price_vs_sma50","price_vs_sma200","sma50_vs_sma200",
        "rsi_14","rsi_7","macd","macd_histogram",
        "stochastic_k","momentum_30d","momentum_90d",
        "bollinger_pct_b","atr_pct","realized_vol_20d","vol_ratio",
        "volume_ratio_10d","obv_trend",
        "relative_strength_vs_spy","beta_60d","correlation_spy_60d",
        "cci_14","mfi_14","williams_r","high_52w_pct","low_52w_pct",
    ] if f in df_tech.columns]
    print(df_tech.set_index("ticker")[show].T.to_string())
elif len(df_tech) > 0:
    print(df_tech.set_index("ticker")[[c for c in fcols if c != "ticker"]].T.to_string())
else:
    print("Run daily_collect.py to populate technical features.")

In [ ]:
if len(df_tech) > 0:
    print("RSI-14  (oversold < 30, overbought > 70):")
    for _, row in df_tech.iterrows():
        rsi = row.get("rsi_14")
        if rsi is not None:
            lvl = "OVERSOLD" if rsi < 30 else ("OVERBOUGHT" if rsi > 70 else "neutral")
            print(f"  {row['ticker']}: {rsi:.1f}  {lvl}")

    print()
    print("Bollinger %B  (0=lower band, 0.5=mid, 1=upper band):")
    for _, row in df_tech.iterrows():
        bb = row.get("bollinger_pct_b")
        if bb is not None:
            pos = "below lower" if bb < 0 else ("above upper" if bb > 1 else f"{bb:.2f}")
            print(f"  {row['ticker']}: %B = {pos}")

    print()
    print("Beta (60d):")
    for _, row in df_tech.iterrows():
        beta = row.get("beta_60d")
        if beta is not None:
            print(f"  {row['ticker']}: {beta:.3f}")

## 7. Fundamental Features — AAPL vs JPM

JPM is a bank — gross_margin, EBITDA, inventory_turnover are not applicable metrics. These nulls are structurally correct and will be imputed to 0 (neutral) by the preprocessor. A bank's "null gross_margin" is not the same as a tech company's null — but both get imputed to 0, which places them at the global mean after z-scoring.

In [ ]:
df_fund = q("""
    SELECT s.ticker, f.*
    FROM fundamental_features f
    JOIN stocks s ON s.id = f.stock_id
    WHERE f.date = (SELECT MAX(date) FROM fundamental_features)
    ORDER BY s.ticker
""")
skip  = {"id","stock_id","date","created_at"}
fcols = [c for c in df_fund.columns if c not in skip]
print(f"Fundamental feature count: {len(fcols)-1}")
print(f"Stocks: {df_fund['ticker'].tolist()}")

In [ ]:
if len(df_fund) > 0:
    groups = {
        "Valuation":            ["pe_ratio","forward_pe","pb_ratio","ps_ratio","ev_ebitda",
                                  "earnings_yield","fcf_yield","price_to_fcf","ev_revenue"],
        "Profitability":        ["roe","roa","roic","gross_margin","operating_margin",
                                  "net_margin","ebitda_margin","fcf_margin"],
        "Quality/Balance Sheet":["piotroski_f_score","altman_z_score",
                                  "debt_to_equity","current_ratio","quick_ratio",
                                  "net_debt_ebitda","interest_coverage"],
        "Growth":               ["revenue_growth_yoy","earnings_growth_yoy","fcf_growth",
                                  "book_value_growth","net_income_growth","buyback_yield"],
    }
    df_set = df_fund.set_index("ticker")
    for grp, features in groups.items():
        present = [f for f in features if f in df_set.columns]
        if not present: continue
        sep55 = "="*55
        print("\n" + sep55)
        print(f"  {grp}")
        print(sep55)
        print(df_set[present].T.to_string())

In [ ]:
# Piotroski F-Score breakdown
p_cols = [c for c in df_fund.columns if c.startswith("p_")]
if p_cols and len(df_fund) > 0:
    labels = {
        "p_positive_net_income":      "Positive net income",
        "p_positive_operating_cf":    "Positive operating CF",
        "p_roa_increasing":           "ROA improving YoY",
        "p_cf_greater_than_ni":       "Cash flow > net income",
        "p_debt_ratio_decreasing":    "Debt ratio falling",
        "p_current_ratio_increasing": "Current ratio improving",
        "p_no_new_shares":            "No new share issuance",
        "p_gross_margin_increasing":  "Gross margin improving",
        "p_asset_turnover_increasing":"Asset turnover improving",
    }
    df_p = df_fund.set_index("ticker")[[c for c in p_cols if c in labels]]
    df_p.columns = [labels.get(c, c) for c in df_p.columns]
    print("Piotroski F-Score Components  (1=pass, 0=fail):")
    print(df_p.T.to_string())
    print()
    print("Score: 7-9 = strong, 0-2 = weak")
    for _, row in df_fund.iterrows():
        score = row.get("piotroski_f_score")
        if score is not None:
            qual = "STRONG" if score >= 7 else ("WEAK" if score <= 2 else "neutral")
            print(f"  {row['ticker']}: {int(score)}/9  {qual}")

## 8. Sentiment Features

In [ ]:
df_sent = q("""
    SELECT s.ticker, sent.*
    FROM sentiment_features sent
    JOIN stocks s ON s.id = sent.stock_id
    WHERE sent.date = (SELECT MAX(date) FROM sentiment_features)
    ORDER BY s.ticker
""")

skip  = {"id","stock_id","date","created_at"}
fcols = [c for c in df_sent.columns if c not in skip and c != "ticker"]
print(f"Sentiment feature count: {len(fcols)}")
if len(df_sent) > 0:
    print()
    print(df_sent.set_index("ticker")[fcols].T.to_string())

In [ ]:
# SEC filings summary
df_filings = q("""
    SELECT s.ticker, sf.form_type, COUNT(*) as count,
           SUM(COALESCE(sf.is_negative_8k, 0)) as negative_8k_count
    FROM sec_filings sf
    JOIN stocks s ON s.id = sf.stock_id
    GROUP BY s.ticker, sf.form_type ORDER BY s.ticker, sf.form_type
""")
if len(df_filings) > 0:
    print("SEC filings summary:")
    print(df_filings.to_string(index=False))
else:
    print("No SEC filings yet.")

df_insider = q("""
    SELECT s.ticker, it.transaction_date, it.transaction_type,
           it.shares, it.price_per_share,
           ROUND(COALESCE(it.shares,0)*COALESCE(it.price_per_share,0),0) as dollar_value
    FROM insider_transactions it
    JOIN stocks s ON s.id = it.stock_id
    ORDER BY it.transaction_date DESC LIMIT 15
""")
if len(df_insider) > 0:
    print("\nRecent insider transactions:")
    print(df_insider.to_string(index=False))
else:
    print("\nNo insider transactions yet.")

## 9. Missingness Analysis

Distinguishing:
- **Structural nulls**: bank without gross_margin — correct, imputed to 0 (neutral)
- **Temporary nulls**: new stock not yet collected — shrinks as universe populates
- **Sector-relative nulls**: ROE vs sector avg — null until 3+ peers collected

In [ ]:
def missingness_report(df, title):
    skip  = {"id","stock_id","date","created_at","ticker"}
    fcols = [c for c in df.columns if c not in skip]
    non_null_pct = df[fcols].notna().mean() * 100
    null_count   = df[fcols].isna().sum()
    report = pd.DataFrame({
        "feature":      fcols,
        "non_null_pct": non_null_pct.values.round(1),
        "null_count":   null_count.values,
    }).sort_values("non_null_pct")

    sep60 = "="*60
    print("\n" + sep60)
    print(f"  {title}  —  {len(fcols)} features, {len(df)} rows")
    print(sep60)
    partial = report[report["non_null_pct"] < 100]
    if len(partial):
        print(partial.to_string(index=False))
    else:
        print("  All features 100% non-null.")
    return report

for df, title in [
    (df_tech,  "Technical Features"),
    (df_fund,  "Fundamental Features"),
    (df_sent,  "Sentiment Features"),
]:
    if len(df) > 0:
        missingness_report(df, title)

In [ ]:
# Missingness heatmap — fundamental (most interesting, banks vs tech)
if len(df_fund) > 0:
    skip    = {"id","stock_id","date","created_at","ticker"}
    fcols   = [c for c in df_fund.columns if c not in skip]
    present = df_fund.set_index("ticker")[fcols].notna().astype(int)
    has_null = present.columns[present.min() == 0]

    if len(has_null) > 0:
        fig, ax = plt.subplots(figsize=(max(8, len(has_null)*0.45), max(3, len(df_fund)*1.2)))
        sns.heatmap(
            present[has_null],
            cmap="RdYlGn", vmin=0, vmax=1,
            annot=True, fmt="d",
            linewidths=0.5, ax=ax, cbar=False
        )
        ax.set_title("Fundamental Feature Presence  (1=present, 0=null)\nOnly features with at least one null shown")
        plt.xticks(rotation=45, ha="right", fontsize=8)
        plt.tight_layout()
        plt.show()
    else:
        print("All fundamental features 100% present.")

## 10. Data Coverage Status

In [ ]:
all_active = q("SELECT id, ticker, sector FROM stocks WHERE is_active=1 ORDER BY ticker")

rows = []
for _, stock in all_active.iterrows():
    sid    = int(stock["id"])
    prices = q("SELECT COUNT(*) as n, MAX(date) as latest FROM daily_prices WHERE stock_id=?",
               params=(sid,)).iloc[0]
    tech   = q("SELECT COUNT(*) as n FROM technical_features WHERE stock_id=?",   params=(sid,)).iloc[0]
    fund   = q("SELECT COUNT(*) as n FROM fundamental_features WHERE stock_id=?", params=(sid,)).iloc[0]
    sent   = q("SELECT COUNT(*) as n FROM sentiment_features WHERE stock_id=?",   params=(sid,)).iloc[0]
    rows.append({
        "ticker":      stock["ticker"],
        "sector":      stock["sector"],
        "prices":      int(prices["n"]),
        "latest":      prices["latest"] or "—",
        "tech":        int(tech["n"]),
        "fund":        int(fund["n"]),
        "sent":        int(sent["n"]),
    })

df_cov = pd.DataFrame(rows)
df_cov["complete"] = (df_cov[["prices","tech","fund","sent"]] > 0).all(axis=1)
print(df_cov.to_string(index=False))
print()
pop = int(df_cov["complete"].sum())
tot = len(df_cov)
print(f"Fully populated: {pop}/{tot} stocks  ({pop/tot:.0%})")

In [ ]:
df_runs = q("""
    SELECT run_date, status, stocks_ok, stocks_failed,
           fund_rows, tech_rows, sent_rows, macro_updated, started_at
    FROM pipeline_runs
    ORDER BY started_at DESC LIMIT 10
""")
if len(df_runs) > 0:
    print("Recent pipeline runs:")
    print(df_runs.to_string(index=False))
else:
    print("No pipeline runs yet.  Run: python scripts/daily_collect.py")

## Summary

### Current state
| Table | Rows | Notes |
|---|---|---|
| `daily_prices` | ~12,500 | AAPL, JPM, SPY + 7 ETFs, 5yr history |
| `technical_features` | 2 | AAPL, JPM — 61 features each |
| `fundamental_features` | 2 | AAPL, JPM — 69 features each |
| `sentiment_features` | 2 | AAPL, JPM — 22 features each |
| `macro_features` | 0 | Needs FRED API key in `.env` |

### What expands with `daily_collect.py`
- All 30 active stocks get prices, technical, fundamental, and sentiment features
- Sector-relative features populate once 3+ stocks per sector exist
- A backfill run (`--date 2024-01-01` through today) creates the ML training history

### Next: Label construction (Week 3)
```python
# 90-day forward return from adj_close
future_price   = adj_close.shift(-90)
forward_return = (future_price / adj_close) - 1

# 4-class labels
# Class 0: < -5%          (big loss)
# Class 1: -5% to +5%     (flat)
# Class 2: +5% to +15%    (moderate gain)
# Class 3: > +15%          (strong gain)
```

Join labeled prices to feature tables on `(stock_id, date)` → XGBoost + LSTM training dataset.